# NB1 — .r3d 추출 (CPU)
Record3D `.r3d` → 세션 입력. **GPU 불필요** (런타임: CPU).

1. 전체 스캔: 모든 프레임의 LiDAR depth 로 *motion envelope* 계산 (NB7 IMU 동기화의 coarse 신호) + 미리보기
2. window 선택 → 그 구간만 `00_extract/` 에 추출 (rgb 원본 jpg, depth/conf npy, intrinsics)

산출: `sessions/{SESSION}/00_extract/` — `meta.json`, `motion_envelope.npz`, `scan_preview.png`, `rgb/`, `depth_npy/`, `conf_npy/`, `intrinsics.json`

| SESSION | 영상(.r3d) | IMU |
|---|---|---|
| `0729_walk` | 002 보행과 같은 시각의 r3d | `MT_01200295_002_60fps_walking` |
| `0729_full` | 001 과 같은 시각의 r3d | `MT_01200295_001_full` |

같은 영상에서 다른 구간(window)을 또 돌리려면 `SESSION` 이름만 바꿔 (`0729_walk_w2` 등) NB1 부터 다시.

In [ ]:
#@title 0. 설정 — 세션마다 여기만 바꾸면 됨
ROOT    = "/content/drive/MyDrive/walking_0729"   #@param {type:"string"}
SESSION = "0729_walk_3kmh"                        #@param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")
import os, sys, importlib
os.environ["WALK_ROOT"] = ROOT
os.environ["WALK_SESSION"] = SESSION
sys.path.insert(0, f"{ROOT}/scripts")
import walkcfg; importlib.reload(walkcfg)
assert os.path.isdir(f"{ROOT}/scripts"), f"{ROOT}/scripts 없음 — colab_kit 업로드 위치 확인"
print("세션 폴더:", walkcfg.SESS)

In [ ]:
#@title 1. r3d 위치 + lzfse 설치
R3D_PATH = "/content/drive/MyDrive/walking_0729/raw/260729_60fps_walkingonly.r3d"   #@param {type:"string"}
# 비워두면 {ROOT}/raw/ 에서 .r3d 목록을 보여줌
!pip -q install lzfse
import glob, shutil
if not R3D_PATH:
    cands = sorted(glob.glob(f"{ROOT}/raw/*.r3d")) + sorted(glob.glob("/content/drive/MyDrive/*.r3d"))
    for c in cands: print(c, f"{os.path.getsize(c)/1e9:.2f} GB")
    assert cands, "r3d 를 {ROOT}/raw/ 에 올리거나 R3D_PATH 지정"
    if len(cands) == 1: R3D_PATH = cands[0]
    else: raise SystemExit("여러 개 → 위 목록에서 골라 R3D_PATH 에 붙여넣고 재실행")
# 드라이브에서 zip 랜덤 접근은 느림 → 로컬로 복사
LOCAL_R3D = "/content/" + os.path.basename(R3D_PATH)
if not os.path.exists(LOCAL_R3D):
    shutil.copy(R3D_PATH, LOCAL_R3D)
print("r3d:", LOCAL_R3D, f"{os.path.getsize(LOCAL_R3D)/1e9:.2f} GB")

## 3. window 고르기
- 위 곡선에서 **보행이 안정된 구간** (트레드밀 오르내림·속도 전환 직후 제외) 을 고른다.
- 4/20 은 250프레임 (30fps ≈ 8.3s, 약 6 보행주기). fps 가 60 이면 같은 프레임 수가 ~4s.
- **드라이브 용량**: NB2 산출이 프레임당 약 40~45MB (`work/out` + `work/video`, NB3 `.bak` 포함). 250f ≈ 11GB.
- IMU(7/29 002) 기준 타임라인: ~27s 보행 시작 → ~45s 속도↑ → ~98s 속도↑ (IMU 시계). 영상 곡선에서도 같은 계단이 보여야 함.

In [ ]:
#@title 2. 전체 스캔 (motion envelope + 미리보기)
!cd "{ROOT}/scripts" && python extract_session.py --r3d "{LOCAL_R3D}" --scan
from IPython.display import Image, display
import json
display(Image(f"{walkcfg.EXTRACT_DIR}/scan_preview.png"))
meta = json.load(open(f"{walkcfg.EXTRACT_DIR}/meta.json"))
print(json.dumps({k: meta[k] for k in ("r3d","r3d_start_local","fps","time_source","n_frames","duration_s")}, indent=1, ensure_ascii=False))

# 2-B fid 표시 미리보기 영상

In [ ]:
VFROM = None   # 시작 fid (None = 처음부터)
VTO   = None   # 끝 fid   (None = 끝까지)
WIDTH = 480

import zipfile, json, os, subprocess, cv2, numpy as np
z = zipfile.ZipFile(LOCAL_R3D)
meta = json.load(z.open("metadata"))
fps = float(meta.get("fps", 30))
ts = meta.get("frameTimestamps")
fids = sorted(int(n[5:-4]) for n in z.namelist() if n.startswith("rgbd/") and n.endswith(".jpg"))
t0 = ts[fids[0]] if ts else 0
sel = [f for f in fids if (VFROM is None or f >= VFROM) and (VTO is None or f <= VTO)]

mp = f"{walkcfg.EXTRACT_DIR}/meta.json"
win = json.load(open(mp)).get("window", {}) if os.path.exists(mp) else {}
w0, w1 = win.get("start", -1), win.get("end", -2)

im0 = cv2.imdecode(np.frombuffer(z.read(f"rgbd/{sel[0]}.jpg"), np.uint8), cv2.IMREAD_COLOR)
H = int(round(im0.shape[0] * WIDTH / im0.shape[1] / 2) * 2)
OUT = f"{walkcfg.EXTRACT_DIR}/preview_fid.mp4"
p = subprocess.Popen(["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "bgr24",
                      "-s", f"{WIDTH}x{H}", "-r", f"{fps:.3f}", "-i", "-",
                      "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "26", OUT], stdin=subprocess.PIPE)
for i, f in enumerate(sel):
    im = cv2.imdecode(np.frombuffer(z.read(f"rgbd/{f}.jpg"), np.uint8), cv2.IMREAD_COLOR)
    im = cv2.resize(im, (WIDTH, H), interpolation=cv2.INTER_AREA)
    t = (ts[f] - t0) if ts else f / fps
    cv2.rectangle(im, (0, 0), (WIDTH, 34), (0, 0, 0), -1)
    cv2.putText(im, f"fid {f:05d}  t={t:7.2f}s", (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    if w0 <= f <= w1:
        cv2.rectangle(im, (2, 2), (WIDTH - 3, H - 3), (0, 220, 0), 6)
        cv2.putText(im, "EXTRACTED", (8, H - 14), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 220, 0), 2)
    p.stdin.write(im.tobytes())
    if i % 1000 == 0: print(f"{i}/{len(sel)}")
p.stdin.close(); p.wait()
print("저장:", OUT)

In [ ]:
#@title 3. 선택 구간 추출
START_FID = 5100     #@param {type:"integer"}
N_FRAMES  = 250   #@param {type:"integer"}
STEP      = 2     #@param {type:"integer"}
# 60fps 영상은 STEP = 2 (2프레임마다 1장 → 30fps, 파이프라인 파라미터가 30fps 기준)
!cd "{ROOT}/scripts" && python extract_session.py --r3d "{LOCAL_R3D}" --start {START_FID} --n {N_FRAMES} --step {STEP}

In [ ]:
#@title 4. 확인 (첫 프레임 RGB / depth / conf)
import numpy as np, matplotlib.pyplot as plt, glob, json
E = walkcfg.EXTRACT_DIR
rg = sorted(glob.glob(f"{E}/rgb/*.jpg")); dp = sorted(glob.glob(f"{E}/depth_npy/*.npy")); cf = sorted(glob.glob(f"{E}/conf_npy/*.npy"))
print(f"rgb {len(rg)} | depth {len(dp)} | conf {len(cf)}")
print("intrinsics:", json.load(open(f"{E}/intrinsics.json")))
d = np.load(dp[0]); c = np.load(cf[0])
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(plt.imread(rg[0])); ax[0].set_title(os.path.basename(rg[0]))
im = ax[1].imshow(np.where(d > 0, d, np.nan), cmap="turbo"); plt.colorbar(im, ax=ax[1]); ax[1].set_title(f"depth (m) median {np.median(d[d>0]):.2f}")
ax[2].imshow(c, vmin=0, vmax=2); ax[2].set_title(f"conf uniq {np.unique(c)}")
plt.show()

## 시각화 (NB1)

In [ ]:
#@title 시각화 — NB1 결과 확인 (viz/NB1/  + viz/videos/)
FIDS = ""   #@param {type:"string"}
N_EVEN = 4           #@param {type:"integer"}
# 셀 0 을 안 돌렸어도 혼자 동작하도록 (드라이브 마운트 + 세션 설정)
import os, sys, importlib
if "walkcfg" not in globals():
    ROOT = "/content/drive/MyDrive/walking_0729"   # 셀 0 과 같게
    SESSION = "0729_full_3kmh"                     # 셀 0 과 같게
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["WALK_ROOT"] = ROOT; os.environ["WALK_SESSION"] = SESSION
    sys.path.insert(0, f"{ROOT}/scripts"); import walkcfg
print("세션:", walkcfg.SESS)
MAKE_VIDEO = True    #@param {type:"boolean"}
FPS = 15             #@param {type:"number"}
!pip -q install imageio-ffmpeg
!cd "{ROOT}/scripts" && python viz_stages.py --nb 1 --fids {FIDS} --n {N_EVEN}
if MAKE_VIDEO:
    !cd "{ROOT}/scripts" && python viz_video.py --nb 1 --fps {FPS}
from IPython.display import Image, display
import glob, os
for p in sorted(glob.glob(f"{walkcfg.SESS}/viz/NB1/*.png")):
    print(os.path.basename(p)); display(Image(p, width=1100))
from IPython.display import Video
_mp4 = f"{walkcfg.SESS}/viz/videos/NB1_raw.mp4"
if os.path.exists(_mp4):
    print("영상:", _mp4, f"{os.path.getsize(_mp4)/1e6:.1f} MB  (노트북에 넣어 재생 — 확인 후 출력 지우기 권장)")
    display(Video(_mp4, embed=True, width=1100))